## Exercise 12.1 (Clustering of univariate time series)  

- Load the GesturePebbleZ1 data set. Note that it is a variable-length time series dataset and you need to either use ret_type="df-list" or ret_type="nested_univ" to load it (using the example above).
- Bring the data into a suitable format and explore the data set visually (using plots).
- Use the Time Series k-means algorithm to cluster the data. Do you need to reshape the data for this algorithm?
- Use the KASBA Holder and Bagnall (2026) algorithm to cluster the data (e.g. implemented in aeon).
- Include useful (!) visualizations.
- How do the results compare?
- After the exercise, you should be able to answer the following questions: 
    - How does KASBA differ from the commonly used k-means algorithm? 
    - In you will find MSM defined. You should be able to illustrate how MSM is calculated.

# 1 Load the GesturePebbleZ1 data set. Note that it is a variable-length time series dataset and you need to either use ret_type="df-list" or ret_type="nested_univ" to load it (using the example above).

In [1]:
from sktime.datasets import load_UCR_UEA_dataset

# Datensatz und Format entsprechend dem Hinweis im Skript festlegen.
ds_name = "GesturePebbleZ1"
ret_type = "df-list"

# Trainingsdaten laden. Die Liste erlaubt unterschiedlich lange Zeitreihen.
X_train, y_train = load_UCR_UEA_dataset(
    name=ds_name,
    split="train",
    return_type=ret_type,
)

# Testdaten im gleichen Format laden.
X_test, y_test = load_UCR_UEA_dataset(
    name=ds_name,
    split="test",
    return_type=ret_type,
)

print("Anzahl Trainingszeitreihen:", len(X_train))
print("Anzahl Testzeitreihen:", len(X_test))

c:\Users\dbros\Documents\Master Mechatronik MCI\3. Semester\Maschinelles_lernen_in_der_Datenverarbeitung\.venv-session3\Lib\site-packages\sktime\datasets\_dataset_downloader.py:422: UserWarning: DatasetDownloader skipping HuggingFaceDownloader as it requires huggingface-hub. We recommend this location as download mirror no. 1, to use this please install huggingface-hub, by pip install huggingface-hub.
  warnings.warn(


Anzahl Trainingszeitreihen: 132
Anzahl Testzeitreihen: 172


# 2 Bring the data into a suitable format and explore the data set visually (using plots).

In [2]:
import numpy as np
import plotly.graph_objects as go

# Trainings- und Testdaten zusammenführen.
# Bei df-list werden die beiden Listen mit + verbunden.
X = X_train + X_test
y = np.concatenate((y_train, y_test))

print("Anzahl Zeitreihen:", len(X))

# Alle Zeitreihen darstellen.
fig = go.Figure()

for series in X:
    fig.add_trace(
        go.Scatter(
            y=series.iloc[:, 0].to_numpy(),
            mode="lines",
            line={"width": 1, "color": "grey"},
            opacity=0.2,
            showlegend=False,
        )
    )

fig.update_layout(
    title="All Time Series in X",
    xaxis_title="Time",
    yaxis_title="Value",
    height=400,
)

fig.show()

# Ausgewählte Zeitreihen darstellen, wie im Skript.
COLORS = ["#A6CEE3", "#B2DF8A", "#FDBF6F", "#CAB2D6"]
sampled_ids = [0, 10, 25, 81]

fig = go.Figure()

for idx, i in enumerate(sampled_ids):
    fig.add_trace(
        go.Scatter(
            y=X[i].iloc[:, 0].to_numpy(),
            mode="lines",
            line={"width": 3, "color": COLORS[idx]},
            name=f"Sample {i}",
        )
    )

fig.update_layout(
    title="Sampled Time Series from X",
    xaxis_title="Time",
    yaxis_title="Value",
    height=400,
)

fig.show()

Anzahl Zeitreihen: 304


# 3 Use the Time Series k-means algorithm to cluster the data. Do you need to reshape the data for this algorithm?

In [3]:
import time
from tslearn.preprocessing import TimeSeriesResampler
from sktime.clustering.k_means import TimeSeriesKMeansTslearn

# 1. Messwerte aus den einzelnen DataFrames übernehmen.
series_list = [series.iloc[:, 0].to_numpy() for series in X]

# 2. Gemeinsame Länge festlegen.
# Die mittlere Länge nach Median dient hier als Kompromiss.
target_length = int(np.median([len(series) for series in series_list]))

# Alle Reihen durch Interpolation auf diese Länge bringen.
X_resampled = TimeSeriesResampler(
    sz=target_length
).fit_transform(series_list)

# 3. Univariate Reihen als zweidimensionale Matrix bereitstellen.
# Zeilen: Zeitreihen, Spalten: Messpunkte.
X_kmeans = X_resampled[:, :, 0]

print("Datenform für k-Means:", X_kmeans.shape)



# 4. Clusteranzahl entsprechend der bekannten Klassenanzahl wählen.
# Analog zum Skript; die Klassenlabels werden nicht mittrainiert.
k = len(np.unique(y))                   # Clusteranzahl = Anzahl Klassenlabels

# KMeans-Clusterer mit tslearn erstellen.
clusterer = TimeSeriesKMeansTslearn(
    n_clusters=k,
    metric="euclidean",
    random_state=42
)

# 5. Clustering durchführen und Laufzeit messen.
start_time = time.time()

y_predicted = clusterer.fit_predict(X_kmeans)       # Clusterzuordnungen für die Zeitreihen ermitteln.

duration = time.time() - start_time                 # Laufzeit in Sekunden berechnen.
#
print("Anzahl Cluster:", k)
print(f"Clustering dauerte {duration:.2f} Sekunden.")

c:\Users\dbros\Documents\Master Mechatronik MCI\3. Semester\Maschinelles_lernen_in_der_Datenverarbeitung\.venv-session3\Lib\site-packages\tslearn\bases\bases.py:16: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


Datenform für k-Means: (304, 190)
Anzahl Cluster: 6
Clustering dauerte 0.57 Sekunden.


# 4 Use the KASBA Holder and Bagnall (2026) algorithm to cluster the data (e.g. implemented in aeon).

In [4]:
from aeon.clustering import KASBA

# 1. KASBA mit derselben Clusteranzahl wie k-Means konfigurieren.
# MSM ist das standardmäßig verwendete Distanzmaß von KASBA.
clusterer_kasba = KASBA(
    n_clusters=k,
    distance="msm",
    random_state=42
)

# 2. Dieselben vorbereiteten Zeitreihen wie bei k-Means verwenden.
# X_kmeans enthält eine Zeitreihe pro Zeile.
start_time = time.time()

y_predicted_kasba = clusterer_kasba.fit_predict(X_kmeans)

# 3. Laufzeit und Anzahl der erzeugten Cluster ausgeben.
duration_kasba = time.time() - start_time

print("Anzahl erzeugter Cluster:", len(set(y_predicted_kasba)))
print(f"KASBA-Clustering dauerte {duration_kasba:.2f} Sekunden.")

Anzahl erzeugter Cluster: 6
KASBA-Clustering dauerte 3.71 Sekunden.


# 5 Include useful (!) visualizations.

In [6]:
# Sechs Farben für unsere sechs Cluster.
COLORS = [
    "#A6CEE3", "#B2DF8A", "#FDBF6F",
    "#CAB2D6", "#FB9A99", "#FFFF99"
]


# Darstellung der Cluster entsprechend dem Skript.
def plot_clusters(X, y, title):
    fig = go.Figure()

    for cluster_idx, cluster in enumerate(sorted(set(y))):
        idx = np.where(y == cluster)[0]
        show_legend = True

        for i in idx:
            fig.add_trace(
                go.Scatter(
                    y=X[i, :, 0],
                    mode="lines",
                    line={"width": 1, "color": COLORS[cluster_idx]},
                    opacity=0.5,
                    name=f"Cluster {chr(ord('A') + cluster_idx)}",
                    legendgroup=str(cluster_idx),
                    showlegend=show_legend,
                )
            )
            show_legend = False

    fig.update_layout(
        title=title,
        xaxis_title="Messpunkt nach Interpolation",
        yaxis_title="Messwert",
        height=500,
    )

    fig.show()


# Darstellung der Clusterzentren entsprechend dem Skript.
# Hier übergeben wir die Zentren direkt als zweidimensionale Matrix.
def plot_centroids(centers, title):
    fig = go.Figure()

    for cluster_idx, centroid in enumerate(centers):
        fig.add_trace(
            go.Scatter(
                y=centroid,
                mode="lines",
                line={"width": 3, "color": COLORS[cluster_idx]},
                name=f"Zentrum {chr(ord('A') + cluster_idx)}",
            )
        )

    fig.update_layout(
        title=title,
        xaxis_title="Messpunkt nach Interpolation",
        yaxis_title="Messwert",
        height=400,
    )

    fig.show()


# 1. Zeitreihen nach den gefundenen Clustern einfärben.
# X_resampled enthält dieselben Daten wie X_kmeans,
# zusätzlich mit einer Achse für die einzige Messgröße.
plot_clusters(
    X_resampled,
    y_predicted,
    "Time Series k-Means – Cluster"
)

plot_clusters(
    X_resampled,
    y_predicted_kasba,
    "KASBA – Cluster"
)

# 2. Clusterzentren beider Verfahren darstellen.
# sktime/tslearn: (Cluster, Messpunkte, Messgrößen)
centers_kmeans = clusterer.cluster_centers_[:, 0, :]

# aeon: (Cluster, Messgrößen, Messpunkte)
centers_kasba = clusterer_kasba.cluster_centers_[:, 0, :]

plot_centroids(
    centers_kmeans,
    "Time Series k-Means – Clusterzentren"
)

plot_centroids(
    centers_kasba,
    "KASBA – Clusterzentren"
)

# 6 How do the results compare?

In [7]:
import pandas as pd

cluster_names = [f"Cluster {chr(ord('A') + i)}" for i in range(k)]

for name, labels in [
    ("k-Means", y_predicted),
    ("KASBA", y_predicted_kasba),
]:
    table = pd.crosstab(
        pd.Series(y, name="Gestenklasse"),
        pd.Series(labels, name="Cluster")
    ).reindex(columns=range(k), fill_value=0)

    table.columns = cluster_names

    print(name)
    display(table)

k-Means


,Cluster A,Cluster B,Cluster C,Cluster D,Cluster E,Cluster F
Gestenklasse,,,,,,
1,0,0,30,0,19,0
2,0,0,1,0,12,32
3,0,0,0,54,0,0
4,0,0,0,33,19,1
5,1,0,44,0,7,0
6,39,12,0,0,0,0


KASBA


,Cluster A,Cluster B,Cluster C,Cluster D,Cluster E,Cluster F
Gestenklasse,,,,,,
1,0,0,0,0,0,49
2,26,10,0,0,9,0
3,4,3,42,0,5,0
4,22,9,1,1,20,0
5,0,0,0,0,27,25
6,2,13,0,24,12,0


Beide Verfahren gruppieren die 304 Zeitreihen in sechs vorgegebene Cluster. Im gemessenen Lauf war k-Means mit 0,57 Sekunden schneller als KASBA mit 3,71 Sekunden. Diese Laufzeiten beziehen sich auf jeweils einen Durchlauf.

Die Kreuztabellen zeigen bei beiden Verfahren eine teilweise Übereinstimmung mit den bekannten Gestenklassen. Bei k-Means werden alle 54 Zeitreihen der Klasse 3 dem Cluster D zugeordnet. Dieses enthält allerdings zusätzlich 33 Reihen der Klasse 4. Auch die Klassen 1 und 5 werden teilweise im selben Cluster zusammengefasst.

Bei KASBA liegen alle 49 Reihen der Klasse 1 im Cluster F. Dort befinden sich jedoch ebenfalls 25 Reihen der Klasse 5. Klasse 3 wird überwiegend Cluster C zugeordnet, während sich insbesondere die Klassen 2, 4 und 6 auf mehrere Cluster verteilen.

Damit werden die sechs Gestenklassen von keinem Verfahren vollständig getrennt. Eine eindeutig bessere Gruppierung durch KASBA lässt sich aus diesen Ergebnissen nicht ableiten. Die Clusterbezeichnungen sind zwischen den Verfahren beliebig und entsprechen nicht automatisch denselben Gruppen. Außerdem beziehen sich die Ergebnisse auf die auf 190 Messpunkte interpolierten Daten.

# 7 After the exercise, you should be able to answer the following questions: 
- How does KASBA differ from the commonly used k-means algorithm? 
- In you will find MSM defined. You should be able to illustrate how MSM is calculated.

### Wie unterscheidet sich KASBA von gewöhnlichem k-Means?

Beide Verfahren ordnen jede Zeitreihe einem Cluster zu und aktualisieren anschließend die Clusterzentren.

Unser gewöhnliches k-Means verwendet die euklidische Distanz. Dabei werden Werte an gleichen Messpunktpositionen verglichen. Das Zentrum ist der punktweise Mittelwert der zugeordneten Reihen. Zeitlich verschobene, aber ähnlich geformte Verläufe können dadurch als unterschiedlich bewertet werden.

KASBA verwendet hier die elastische MSM-Distanz. Sie ermöglicht einen flexibleren Vergleich zeitlich gestreckter oder verschobener Muster. Die Clusterzentren werden durch ein zur Distanz passendes Baryzentrum-Verfahren angenähert. KASBA beschleunigt die Berechnung unter anderem durch die Dreiecksungleichung und zufällig ausgewählte Teilmengen bei der Aktualisierung der Zentren. Gegenüber einfachem euklidischem k-Means ist es dennoch nicht automatisch schneller.

### Wie wird die MSM-Distanz berechnet?

MSM steht für **Move-Split-Merge**. Die Distanz entspricht den geringsten Gesamtkosten, um eine Zeitreihe durch drei erlaubte Operationen in eine andere umzuwandeln:

- **Move:** Einen Messwert verändern. Die Kosten entsprechen der absoluten Wertänderung.
- **Split:** Einen Messwert duplizieren. Die Kosten betragen einen festgelegten Wert \(c\).
- **Merge:** Zwei benachbarte, gleich große Messwerte zusammenführen. Die Kosten betragen ebenfalls \(c\).

Beispielsweise sollen bei \(c=1\) die Reihen \(X=[1,2,2]\) und \(Y=[1,3]\) verglichen werden:

1. Die beiden benachbarten Zweien zusammenführen: \([1,2,2]\rightarrow[1,2]\). Kosten: \(1\).
2. Den Wert 2 auf 3 ändern: \([1,2]\rightarrow[1,3]\). Kosten: \(|3-2|=1\).

Die Gesamtkosten betragen \(2\). Für dieses Beispiel ist das die MSM-Distanz. Allgemein wird die günstigste Folge der Operationen mithilfe einer Kostenmatrix und dynamischer Programmierung bestimmt.

ipynb exportiert als html mit:

uv run jupyter nbconvert --to html "Tasks/12 Self study - Session 3/Exercise_12.1.ipynb"